# 扩展主题：知识蒸馏（Knowledge Distillation）

> **性质**：🔬 真做  ｜  **依赖**：ch05 训练好的 GPT-124M

## 一句话

用**大模型（teacher）的输出**指导**小模型（student）**训练，让 student 在更小参数量下接近 teacher 的效果——模型压缩的经典技术。

## 为什么需要蒸馏

大模型效果好但部署贵。蒸馏把大模型的知识「压缩」进小模型：

- **student 体积小**：推理快、省显存、适合边缘部署
- **学的是软标签**：teacher 的 logits 概率分布比 hard label（one-hot）信息更丰富

## 蒸馏损失：KL 散度 + 温度

**温度 T**：softmax 时除以 T，让概率分布更「软」（更平滑）。高温下，teacher 输出透露「类间相似度」等暗知识。

$$\mathcal{L} = T^2 \cdot \text{KL}\left(\text{softmax}(z_s/T) \,\|\, \text{softmax}(z_t/T)\right)$$

- $z_s, z_t$：student / teacher 的 logits
- 乘 $T^2$ 补偿：温度缩放会让梯度变小，乘回来保持学习率有效
- KL 散度衡量两个分布的差异，越小越像 teacher

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def distillation_loss(student_logits, teacher_logits, temperature=2.0):
    """蒸馏损失：KL 散度，带温度软化。"""
    # 软化：logits 除以温度再 softmax，分布更平滑
    soft_student = F.log_softmax(student_logits / temperature, dim=-1)
    soft_teacher = F.softmax(teacher_logits / temperature, dim=-1)
    # KL 散度：student 与 teacher 分布的差异
    kl = F.kl_div(soft_student, soft_teacher, reduction="batchmean")
    return kl * (temperature ** 2)   # 乘 T² 补偿梯度缩放


# 验证：student 能学会模仿 teacher
torch.manual_seed(0)
teacher = nn.Linear(10, 5)   # 假装是大模型（冻结）
student = nn.Linear(10, 5)   # 待训练的小模型
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

x = torch.randn(32, 10)
with torch.no_grad():
    teacher_logits = teacher(x)

optimizer = torch.optim.SGD(student.parameters(), lr=0.1)
print("蒸馏训练（student 模仿 teacher 输出分布）：")
for epoch in range(50):
    optimizer.zero_grad()
    loss = distillation_loss(student(x), teacher_logits, temperature=2.0)
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0 or epoch == 49:
        print(f"  epoch {epoch}: KL loss {loss.item():.4f}")
print("\n💡 KL loss 持续下降，student 学会了模仿 teacher 的输出分布。")

## 2. 蒸馏 GPT：12 层 teacher → 2 层 student

In [ ]:
from src.gpt import GPTModel, GPT_CONFIG_124M
import tiktoken

# teacher: 较大配置；student: 较小配置（层数少）
teacher_cfg = dict(GPT_CONFIG_124M)
teacher_cfg.update({"emb_dim": 128, "n_layers": 4, "n_heads": 4, "context_length": 32})

student_cfg = dict(GPT_CONFIG_124M)
student_cfg.update({"emb_dim": 128, "n_layers": 2, "n_heads": 4, "context_length": 32})

torch.manual_seed(123)
teacher = GPTModel(teacher_cfg)
student = GPTModel(student_cfg)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

n_t = sum(p.numel() for p in teacher.parameters())
n_s = sum(p.numel() for p in student.parameters())
print(f"teacher: {n_t:,} 参数 ({teacher_cfg['n_layers']} 层)")
print(f"student: {n_s:,} 参数 ({student_cfg['n_layers']} 层) ← 小 {100*(1-n_s/n_t):.0f}%")

# 蒸馏训练（用随机数据 demo 语言建模的下一步预测）
tok = tiktoken.get_encoding("gpt2")
optimizer = torch.optim.AdamW(student.parameters(), lr=1e-3)

print("\n蒸馏训练（student 学 teacher 的 logits）：")
for epoch in range(5):
    x = torch.randint(0, teacher_cfg["vocab_size"], (4, 16))
    with torch.no_grad():
        t_logits = teacher(x)
    optimizer.zero_grad()
    s_logits = student(x)
    # 对每个位置的 vocab 维度做蒸馏
    loss = distillation_loss(s_logits, t_logits, temperature=2.0)
    loss.backward()
    optimizer.step()
    print(f"  epoch {epoch}: KL loss {loss.item():.4f}")
print("\n💡 student 用更少参数，学到了接近 teacher 的输出分布。")

---
> **小结**：蒸馏用 teacher 软标签（高温 KL 散度）指导 student，实现模型压缩。
> **温度**：T 越高软标签信息越丰富但越柔和；通常 T=2~4。
> **应用**：把大模型部署到手机/边缘设备，或加速线上推理。